<a href="https://colab.research.google.com/github/zerogradlabs/zerograd-python/blob/main/examples/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ZeroGrad quickstart: from `pip install` to the podium

ZeroGrad runs a **championship** of tabular foundation models on your data: each model reads your labeled rows as context, all of them are scored on the same holdout, and the best one becomes your champion. Nothing to train.

A GPU runtime makes it faster (*Runtime → Change runtime type → T4 GPU*), but CPU works too.

In [ ]:
!pip install -q zerograd

## What is there

Every model the SDK knows, with its license and whether you can use it commercially. Non-commercial models are off by default.

In [ ]:
import zerograd
from zerograd import ZeroGrad

zerograd.list_models()

In [ ]:
zerograd.list_tasks()

## Your data

Any DataFrame with a target column works. Here, the breast cancer dataset that ships with scikit-learn: 569 tumors, 30 measurements each, malignant or benign.

In [ ]:
from sklearn.datasets import load_breast_cancer

df = load_breast_cancer(as_frame=True).frame
df["diagnosis"] = df.pop("target").map({0: "malignant", 1: "benign"})

train = df.sample(frac=0.8, random_state=0)
new = df.drop(train.index)
train.shape, new.shape

## The championship

The first run downloads each model's weights from its official Hugging Face repository (once).

In [ ]:
zg = ZeroGrad()
cap = zg.create("breast-cancer", examples=train, target="diagnosis")

In [ ]:
cap.evaluation

In [ ]:
cap.baselines   # the same holdout without a model

## Predict

The champion predicts new rows: the class, its score and each class probability.

In [ ]:
preds = cap.predict(new.drop(columns="diagnosis"))
preds.head()

In [ ]:
(preds["prediction"] == new["diagnosis"]).mean()

## Keep it

A capability saves to one file (no weights inside) and opens anywhere the SDK is installed.

In [ ]:
cap.save("breast-cancer.zg")
cap = zerograd.load("breast-cancer.zg")
cap

## How much context do you need?

A foundation model reads its context (your labeled rows) on every prediction: more rows usually score better and answer slower. `context_curve` fits one model on growing samples, scores each on the same test rows and times it. The recommended size is the smallest one within 0.005 of the best score.

Here on a bigger table, 12,000 rows of the Adult census dataset (income above 50K). It takes a couple of minutes on a CPU.

In [ ]:
from sklearn.datasets import fetch_openml

adult = fetch_openml("adult", version=2, as_frame=True).frame.sample(12_000, random_state=0)
curve = zg.context_curve(adult.iloc[:10_000], "class", test=adult.iloc[10_000:],
                         sizes=[250, 500, 1_000, 2_000, 5_000, 10_000])
curve

In [ ]:
curve.plot();

On a capability, `cap.context_curve(test)` runs the same curve on its champion and its own context, and `cap.update(context_rows=curve.recommended)` keeps the smaller, faster context for predictions.

## Next

* Pick the competitors: `ZeroGrad(models=["tabpfn-v2", "mitra-v2"])` (Mitra-v2 needs `pip install "zerograd[mitra]"`).
* Run it in production with versions, scheduled championships, monitoring and Explain: `ZeroGrad(remote=True)` on [zerograd.dev](https://zerograd.dev).
* Licenses: TabPFN v2 asks products built with it to show *Built with PriorLabs-TabPFN*. See `MODELS.md` and `TERMS.md`.